<a href="https://colab.research.google.com/github/patilprajwal83094-arch/1MS25IS077--ASSIGNMENT-2/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [91]:
import os
import getpass

# Request a valid Gemini API Key starting with 'AIza'
key = getpass.getpass('Paste your Gemini API key (must start with AIza) and press Enter: ')
os.environ['GOOGLE_API_KEY'] = key
print('API key set successfully.')

Paste your Gemini API key (must start with AIza) and press Enter: ··········
API key set successfully.


In [92]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [93]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {


    "artificial_intelligence": (
        "Artificial Intelligence enables machines to perform tasks that normally require human intelligence. "
        "Machine learning is a branch of AI that allows systems to learn from data. "
        "Deep learning uses neural networks to identify complex patterns. "
        "AI is widely used in healthcare, education, finance, and autonomous systems."
    ),


}

In [94]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        # Ensure the content is a string and not an empty placeholder tuple
        if not isinstance(content, str):
            continue

        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))

[Source: artificial_intelligence]
Artificial Intelligence enables machines to perform tasks that normally require human intelligence. Machine learning is a branch of AI that allows systems to learn from data. Deep learning uses neural networks to identify complex patterns. AI is widely used in healthcare, education, finance, and autonomous systems.


In [95]:
# @title Ask gemini to search for something
import google.generativeai as genai
import os

api_key = os.environ.get("GOOGLE_API_KEY", "")

# Configure the SDK to use the API key
genai.configure(api_key=api_key)

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    # Use the correct search tool defined in the notebook
    tools=[search_documents],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

In [96]:
# @title Define the agent loop execution helper
import time
import google.generativeai as genai

def run_agent(question: str):
    """Run a single question through the agent step-by-step loop."""
    print(f"User Question: {question}\n")

    # Send message to model
    chat = model.start_chat(enable_automatic_function_calling=True)
    response = chat.send_message(question)

    # Print final response
    print(f"Agent Response: {response.text}")

In [97]:
import time
from google.api_core.exceptions import GoogleAPICallError

# Updated questions to satisfy cell test validation conditions (>= 4 questions, no default placeholder)
questions = [
    "What is the recommended cooking temperature for medium-rare steak?",
    "How long should a standard pasta dough rest before rolling?",
    "How many ghosts eat carrots everyday?",
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(2)  # Reduced delay to make execution run faster
    print("\n" + "=" * 60)
    try:
        run_agent(q)
    except GoogleAPICallError as e:
        print(f"\n[Error]: Google GenAI API error encountered for this query.\n{e}")
        print("Skipping to the next question...")
    except Exception as e:
        print(f"\n[Unexpected Error]: {e}")
        print("Skipping to the next question...")


User Question: What is the recommended cooking temperature for medium-rare steak?


[Error]: Google GenAI API error encountered for this query.
404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
Skipping to the next question...

User Question: How long should a standard pasta dough rest before rolling?


[Error]: Google GenAI API error encountered for this query.
404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
Skipping to the next question...

User Question: How many ghosts eat carrots everyday?


[Error]: Google GenAI API error encountered for this query.
404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels 

In [98]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)